In [1]:
# Initial setup

import os

# uncomment to disable NVIDIA GPUs
#os.environ['CUDA_VISIBLE_DEVICES'] = ''
# or pick the device (cpu, gpu, and tpu)
#os.environ['JAX_PLATFORMS'] = 'cpu'

# change JAX GPU memory preallocation fraction
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '.95'

# you do not want this
#os.environ['XLA_FLAGS'] = '--xla_gpu_deterministic_ops=true'

In [2]:
# First look at JAX and the environment
import jax
jax.print_environment_info()

jax:    0.6.2
jaxlib: 0.6.2
numpy:  2.2.6
python: 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:45:41) [GCC 13.3.0]
device info: NVIDIA RTX A5000-1, 1 local devices"
process_count: 1
platform: uname_result(system='Linux', node='r903u37n01.grace.ycrc.yale.edu', release='4.18.0-553.52.1.el8_10.x86_64', version='#1 SMP Mon May 5 10:03:48 EDT 2025', machine='x86_64')

$ nvidia-smi
Wed Jun 25 03:14:35 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.133.20             Driver Version: 570.133.20     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|======================

In [3]:
# Initial imports for pmwd

import numpy as np

from pmwd import (Configuration, Cosmology, SimpleLCDM, 
                    particles, boltzmann, linear_power, growth, 
                    white_noise, linear_modes, 
                    lpt, nbody, scatter)

In [4]:
# Determine the box size, grid size and other info 
BoxSize = 1000.0 # Mpc/h
grid = 128 # grid size

# Lagrangian space particle grid spacing
ptcl_spacing = BoxSize / grid
# Particle grid, assume the same as the regular grid 
ptcl_grid_space = (grid, grid, grid)

# Set up configuration
conf = Configuration(ptcl_spacing, ptcl_grid_space, mesh_shape=1) # 1x grid shape 
print(conf)

Configuration(ptcl_spacing=7.8125,
              ptcl_grid_shape=(128, 128, 128),
              mesh_shape=(128, 128, 128),
              cosmo_dtype=dtype('float64'),
              pmid_dtype=dtype('int16'),
              float_dtype=dtype('float32'),
              k_pivot_Mpc=0.05,
              T_cmb=2.7255,
              M=1.98847e+40,
              L=3.0856775815e+22,
              T=3.0856775815e+17,
              transfer_fit=True,
              transfer_fit_nowiggle=False,
              transfer_lgk_min=-4,
              transfer_lgk_max=3,
              transfer_lgk_maxstep=0.0078125,
              growth_rtol=1.4901161193847656e-08,
              growth_atol=1.4901161193847656e-08,
              growth_inistep=(1, None),
              lpt_order=2,
              a_start=0.015625,
              a_stop=1,
              a_lpt_maxstep=0.0078125,
              a_nbody_maxstep=0.015625,
              symp_splits=((0, 0.5), (1, 0.5)),
              chunk_size=16777216)


In [5]:
print(f'Simulating {conf.ptcl_num} particles with a {conf.mesh_shape} mesh for {conf.a_nbody_num} time steps.')

Simulating 2097152 particles with a (128, 128, 128) mesh for 63 time steps.


In [6]:
# Initialize the cosmology 
cosmo = Cosmology(conf, A_s_1e9=2.0, n_s=0.96, h=0.6711, Omega_m=0.3175, Omega_b=0.049)

print(cosmo)

Cosmology(A_s_1e9=Array(2., dtype=float64),
          n_s=Array(0.96, dtype=float64),
          Omega_m=Array(0.318, dtype=float64),
          Omega_b=Array(0.049, dtype=float64),
          h=Array(0.671, dtype=float64),
          Omega_k_=None,
          w_0_=None,
          w_a_=None,
          transfer=None,
          growth=None,
          varlin=None)


In [11]:
cosmo = jax.block_until_ready(boltzmann(cosmo, conf))

In [12]:
print(cosmo)

Cosmology(A_s_1e9=Array(2., dtype=float64),
          n_s=Array(0.96, dtype=float64),
          Omega_m=Array(0.318, dtype=float64),
          Omega_b=Array(0.049, dtype=float64),
          h=Array(0.671, dtype=float64),
          Omega_k_=None,
          w_0_=None,
          w_a_=None,
          transfer=Array([1.000e+00, 9.999e-01, 9.999e-01, 9.999e-01, 9.999e-01, 9.999e-01, 9.999e-01, 9.999e-01, 9.999e-01, 9.999e-01, 9.999e-01,
       9.999e-01, 9.999e-01, 9.999e-01, 9.999e-01, 9.999e-01, 9.998e-01, 9.998e-01, 9.998e-01, 9.998e-01, 9.998e-01, 9.998e-01,
       9.998e-01, 9.998e-01, 9.998e-01, 9.998e-01, 9.998e-01, 9.998e-01, 9.998e-01, 9.998e-01, 9.997e-01, 9.997e-01, 9.997e-01,
       9.997e-01, 9.997e-01, 9.997e-01, 9.997e-01, 9.997e-01, 9.997e-01, 9.997e-01, 9.996e-01, 9.996e-01, 9.996e-01, 9.996e-01,
       9.996e-01, 9.996e-01, 9.996e-01, 9.995e-01, 9.995e-01, 9.995e-01, 9.995e-01, 9.995e-01, 9.994e-01, 9.994e-01, 9.994e-01,
       9.994e-01, 9.994e-01, 9.993e-01, 9.993e-01, 9.

In [13]:
seed = 1

In [14]:
# White noise 
# As we'll see below, 
modes = white_noise(seed, conf)

In [15]:
print(type(modes))
print(modes.shape)
print(modes.dtype)


<class 'jaxlib._jax.ArrayImpl'>
(128, 128, 65)
complex64


In [16]:
# Linear modes 
modes = linear_modes(modes, cosmo, conf)

In [17]:
print(type(modes))
print(modes.shape)
print(modes.dtype)

<class 'jaxlib._jax.ArrayImpl'>
(128, 128, 65)
complex64


In [18]:
# Set up LPT, and particle positions 
ptcl, obsvbl = jax.block_until_ready(lpt(modes, cosmo, conf))

In [21]:
# We don't use observable here.
print(obsvbl)

None


In [26]:
# Particles 
print(type(ptcl))
print(type(ptcl.pmid))
print(ptcl.pmid.shape)
print(ptcl.pmid.devices())

<class 'pmwd.particles.Particles'>
<class 'jaxlib._jax.ArrayImpl'>
(2097152, 3)
{CudaDevice(id=0)}


In [30]:
ptcl.pos()

Array([[999.933, 999.951, 999.904],
       [999.972, 999.977,   7.68 ],
       ...,
       [992.139, 992.207, 984.421],
       [992.159, 992.229, 992.193]], dtype=float64)

In [ ]:
# Print displacements
# Note that all arrays are JAX arrays 
print(type(ptcl.disp))
ptcl.disp

<class 'jaxlib._jax.ArrayImpl'>


Array([[-0.067, -0.049, -0.096],
       [-0.028, -0.023, -0.132],
       ...,
       [-0.049,  0.02 ,  0.046],
       [-0.028,  0.041,  0.006]], dtype=float32)

In [33]:
ptcl, obsvbl = nbody(ptcl, obsvbl, cosmo, conf)

In [34]:
ptcl.pos()

Array([[9.990e+02, 9.978e+02, 9.951e+02],
       [5.782e-01, 4.154e-01, 2.236e+00],
       ...,
       [9.919e+02, 9.946e+02, 9.858e+02],
       [9.923e+02, 9.942e+02, 9.891e+02]], dtype=float64)

In [35]:
ptcl.pmid

Array([[  0,   0,   0],
       [  0,   0,   1],
       ...,
       [127, 127, 126],
       [127, 127, 127]], dtype=int16)